# Satellite Land Use Classification (Transfer Learning)

## Overview

An image classifier that identifies land use and land cover type from
satellite imagery, built by fine-tuning a pre-trained CNN (transfer
learning) instead of training a network from scratch

## Dataset

EuroSAT (Sentinel-2 satellite imagery, ESA) — 27,000 RGB images (64x64),
10 land use classes: AnnualCrop, Forest, HerbaceousVegetation, Highway,
Industrial, Pasture, PermanentCrop, Residential, River, SeaLake.

In [2]:
import torchvision
# torch.utils.data.random_split lets us split a Dataset object into
# non-overlapping subsets of a given size
from torch.utils.data import random_split
# torchvision.transforms provides image preprocessing operations
# (resize, normalization, augmentation) applied before feeding images to a model
from torchvision import transforms
import torch

torch.manual_seed(42)

# EuroSAT images are 64x64 pixels, but the pre-trained ResNet model expects
# 224x224 inputs (the resolution it was originally trained on) — this
# transform pipeline is applied to BOTH train and test images
base_transform = transforms.Compose([
    # Resize every image from 64x64 up to 224x224, matching what the
    # pre-trained model expects as input
    transforms.Resize((224, 224)),
    # Convert the PIL image into a PyTorch tensor (and rescales pixel
    # values from the 0-255 range down to 0-1)
    transforms.ToTensor(),
    # Normalize each of the 3 color channels using the exact mean and
    # standard deviation ImageNet was trained with — required for the
    # pre-trained model to interpret pixel values correctly
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Re-download (or reuse the already-downloaded) dataset, this time passing
# our transform pipeline so every image is preprocessed automatically
# when loaded
full_dataset = torchvision.datasets.EuroSAT(root='./data', download=True, transform=base_transform)

# Compute how many images go into training (80%) vs testing (20%)
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size

# Randomly split the full dataset into two non-overlapping subsets of the
# sizes just computed — NOTE: this is a random (non-stratified) split,
# we'll verify class balance afterward
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

print(f"Train size: {len(train_dataset)}")
print(f"Test size: {len(test_dataset)}")

Train size: 21600
Test size: 5400


In [8]:
# Counter is a convenient dictionary-like object for counting occurrences
# of items — here, we'll use it to count how many images of each class
# ended up in each split
from collections import Counter

# full_dataset.targets holds the integer label of every image in the
# original, unsplit dataset, in the same order as full_dataset itself
all_labels = full_dataset.targets

# train_dataset.indices holds the original positions (from full_dataset)
# of the images that random_split assigned to the training set —
# we use these indices to look up each training image's true label
train_labels = [all_labels[i] for i in train_dataset.indices]

# Same logic for the test set: look up the true label of each image
# using the indices random_split assigned to the test subset
test_labels = [all_labels[i] for i in test_dataset.indices]

# Counter(train_labels) tallies how many times each class label appears
# in the training set, giving us a class -> count mapping
train_counts = Counter(train_labels)
test_counts = Counter(test_labels)

# Loop over each class index and its human-readable name together,
# printing the train/test count for that specific class
for class_idx, class_name in enumerate(full_dataset.classes):
    print(f"{class_name + ':':<30}train={train_counts[class_idx]}, test={test_counts[class_idx]}")

print("\n"*3)

# Compute the total number of images in each split, needed to turn
# raw counts into percentages
total_train = len(train_dataset)
total_test = len(test_dataset)

# For each class, calculate what percentage of the training set and what
# percentage of the test set it represents — comparing these two
# percentages tells us whether the random split preserved class balance
for class_idx, class_name in enumerate(full_dataset.classes):
    train_pct = train_counts[class_idx] / total_train * 100
    test_pct = test_counts[class_idx] / total_test * 100
    print(f"{class_name + ':':<30}train={train_pct:5.1f}%, test={test_pct:5.1f}%")

AnnualCrop:                   train=2369, test=631
Forest:                       train=2418, test=582
HerbaceousVegetation:         train=2388, test=612
Highway:                      train=1975, test=525
Industrial:                   train=2016, test=484
Pasture:                      train=1604, test=396
PermanentCrop:                train=1994, test=506
Residential:                  train=2394, test=606
River:                        train=2003, test=497
SeaLake:                      train=2439, test=561




AnnualCrop:                   train= 11.0%, test= 11.7%
Forest:                       train= 11.2%, test= 10.8%
HerbaceousVegetation:         train= 11.1%, test= 11.3%
Highway:                      train=  9.1%, test=  9.7%
Industrial:                   train=  9.3%, test=  9.0%
Pasture:                      train=  7.4%, test=  7.3%
PermanentCrop:                train=  9.2%, test=  9.4%
Residential:                  train= 11.1%, test= 11.2%
River:                        train=  

# ResNet time!

In [10]:
# torch.nn contains building blocks for neural networks (layers, loss
# functions, etc.) — we need it here to define our replacement final layer
import torch.nn as nn
# torchvision.models provides architectures with optional pre-trained
# weights, including ResNet
from torchvision import models

# Load the ResNet18 architecture with weights pre-trained on ImageNet
# (1.2 million images, 1000 categories) — this downloads the pre-trained
# weights the first time it's run
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Loop over every parameter (weight and bias) in the entire pre-trained
# network, and freeze each one by setting requires_grad to False —
# this tells PyTorch's autograd to NOT compute gradients for these
# parameters, so they will never be updated during training
for param in model.parameters():
    param.requires_grad = False

# model.fc is ResNet's final fully-connected layer, originally sized to
# output 1000 values (one per ImageNet class). model.fc.in_features
# tells us how many input features that layer expects (512 for ResNet18) —
# we reuse this number rather than hardcoding it, so the code stays
# correct even if we later switch to a different ResNet variant
num_features = model.fc.in_features

# Replace the final layer entirely with a brand new nn.Linear layer,
# going from the same number of input features to just 10 outputs
# (one per EuroSAT class). This new layer is created fresh, so its
# weights start randomly initialized and DO have requires_grad=True
# by default — this is the only part of the network that will actually
# learn during training
model.fc = nn.Linear(num_features, 10)

# Select GPU if available, otherwise fall back to CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Move all of the model's parameters (both frozen and trainable) to
# the selected device
model = model.to(device)

# Print a summary confirming how many parameters are trainable
# vs frozen — a useful sanity check that the freezing worked correctly
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,}")
print("\t\t# 512x10 + 10 trainable parameters: one weight per input feature" \
      "(512) for each of the 10 output classes, plus one bias term per class.")
print(f"Total parameters: {total_params:,}")

Trainable parameters: 5,130
		# 512x10 + 10 trainable parameters: one weight per input feature(512) for each of the 10 output classes, plus one bias term per class.
Total parameters: 11,181,642


# Training

In [11]:
# DataLoader wraps a Dataset and handles batching, shuffling, and
# efficient loading of data during training/evaluation
from torch.utils.data import DataLoader
# optim contains optimization algorithms (Adam, SGD, etc.) used to
# update model weights based on computed gradients
import torch.optim as optim

# Wrap the training subset in a DataLoader: batch_size=32 groups 32
# images together per training step, shuffle=True randomizes the order
# of images every epoch (important so the model doesn't learn from a
# fixed sequence)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Wrap the test subset in a DataLoader too: same batch size, but
# shuffle=False since evaluation order doesn't matter and we want
# reproducible, consistent results
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# CrossEntropyLoss is the standard loss function for multi-class
# classification — it internally applies softmax to the model's raw
# outputs and compares the resulting probability distribution to the
# true class label
criterion = nn.CrossEntropyLoss()

# Adam optimizer will update model weights during training.
# model.parameters() would normally pass ALL parameters, but since we
# froze most of them (requires_grad=False), PyTorch's optimizer will
# only actually update the ~5,130 unfrozen ones — freezing at the
# parameter level, not by filtering what we pass here, is what makes
# transfer learning efficient
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We'll train for 5 epochs since we're only training a small final layer on
# top of already-strong pre-trained features, so convergence should be much faster
n_epochs = 5

for epoch in range(n_epochs):
    # Set the model to training mode (affects layers like BatchNorm
    # and Dropout, though ResNet18's frozen BatchNorm layers will stay
    # in their pre-trained state regardless)
    model.train()

    # Reset per-epoch accumulators: total loss, count of correct
    # predictions, and total number of examples processed
    running_loss = 0.0
    correct = 0
    total = 0

    # Iterate over the training data one batch at a time
    for images, labels in train_loader:
        # Move this batch's images and labels to the same device
        # (GPU or CPU) the model lives on
        images, labels = images.to(device), labels.to(device)

        # Clear gradients accumulated from the previous batch before
        # computing new ones
        optimizer.zero_grad()

        # Forward pass: run the batch through the network, producing
        # 10 raw logits (one per class) for each image
        outputs = model(images)

        # Compute how far the model's predictions are from the true
        # labels for this batch
        loss = criterion(outputs, labels)

        # Backpropagation: compute the gradient of the loss with
        # respect to every trainable parameter (only the unfrozen
        # final layer will actually receive non-zero gradients)
        loss.backward()

        # Apply the weight update using the gradients just computed
        optimizer.step()

        # Accumulate this batch's loss value (as a plain number, via
        # .item()) to compute the average loss for the epoch later
        running_loss += loss.item()

        # For each image, find which of the 10 output logits is
        # highest — that's the model's predicted class
        _, predicted = torch.max(outputs, 1)

        # Add this batch's size to the running total of images seen
        total += labels.size(0)

        # Compare predictions to true labels, count how many matched,
        # and add that count to the running total of correct predictions
        correct += (predicted == labels).sum().item()

    # After processing all batches in this epoch, compute the average
    # loss per batch and the overall training accuracy
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total

    # Print this epoch's results so we can monitor training progress
    print(f"Epoch {epoch+1}/{n_epochs} - Loss: {epoch_loss:.4f} - Train Accuracy: {epoch_acc:.4f}")

Epoch 1/5 - Loss: 0.5664 - Train Accuracy: 0.8309
Epoch 2/5 - Loss: 0.3130 - Train Accuracy: 0.8967
Epoch 3/5 - Loss: 0.2801 - Train Accuracy: 0.9057
Epoch 4/5 - Loss: 0.2628 - Train Accuracy: 0.9100
Epoch 5/5 - Loss: 0.2445 - Train Accuracy: 0.9177


# Test Set

In [12]:
# scikit-learn metrics for evaluating multi-class classification —
# accuracy_score gives overall accuracy, classification_report gives
# per-class precision/recall/F1, confusion_matrix shows exactly which
# classes get confused with which
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Switch the model to evaluation mode — disables training-specific
# behaviors (relevant if any Dropout/BatchNorm layers were unfrozen,
# though here all frozen layers already behave consistently)
model.eval()

# Lists to accumulate predictions and true labels across all test
# batches, so we can compute metrics over the entire test set at once
all_preds = []
all_labels = []

# Disable gradient tracking for this block — we're only doing inference
# (no backward pass), so this saves memory and speeds up computation
with torch.no_grad():
    # Iterate over the test data one batch at a time
    for images, labels in test_loader:
        # Move this batch to the same device as the model
        images, labels = images.to(device), labels.to(device)

        # Forward pass only — get the model's raw output logits
        outputs = model(images)

        # Pick the class with the highest logit for each image —
        # that's the model's predicted class
        _, predicted = torch.max(outputs, 1)

        # Move predictions and true labels back to CPU and convert to
        # NumPy arrays (required by scikit-learn), then append them to
        # our running lists
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Compute overall accuracy across the entire test set
test_acc = accuracy_score(all_labels, all_preds)
print(f"Test Accuracy: {test_acc:.4f}")

# classification_report prints precision, recall, and F1-score for
# EACH of the 10 classes individually, plus overall averages —
# target_names maps the numeric class indices back to their readable
# names for a more interpretable printout
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=full_dataset.classes))

# The confusion matrix shows, for every true class (rows) how many
# images were predicted as each possible class (columns) — the
# diagonal shows correct predictions, off-diagonal values show
# specific misclassifications
print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

Test Accuracy: 0.9319

Classification Report:
                      precision    recall  f1-score   support

          AnnualCrop       0.96      0.93      0.94       631
              Forest       0.98      0.97      0.97       582
HerbaceousVegetation       0.92      0.93      0.93       612
             Highway       0.88      0.86      0.87       525
          Industrial       0.94      0.96      0.95       484
             Pasture       0.89      0.95      0.92       396
       PermanentCrop       0.91      0.90      0.90       506
         Residential       0.97      0.97      0.97       606
               River       0.90      0.84      0.87       497
             SeaLake       0.95      0.99      0.97       561

            accuracy                           0.93      5400
           macro avg       0.93      0.93      0.93      5400
        weighted avg       0.93      0.93      0.93      5400

Confusion Matrix:
[[586   0   2   9   1   8  14   0   4   7]
 [  0 565   3   0   0 

## Summary

A ResNet18 pre-trained on ImageNet was adapted to classify satellite land
use imagery (EuroSAT, 10 classes) using transfer learning: all pre-trained
layers were frozen, and only a new final classification layer was trained
from scratch — 5,130 trainable parameters out of 11.2 million total (0.046%
of the network).

Despite training only this tiny fraction of the model, and for just 5
epochs, the model reached **93.19% test accuracy**, demonstrating the core
value of transfer learning: general visual features learned from millions
of natural photographs (edges, textures, shapes) transferred effectively to
a visually very different domain — 64x64 satellite imagery upscaled to
224x224 — without needing to relearn them from scratch.

Per-class performance was fairly uniform (macro and weighted averages both
0.93), with two notable exceptions: **Highway** and **River** were the
hardest classes to distinguish from each other (27 and 38 misclassifications
respectively, out of their confusion matrix rows) — both tend to appear as
long, narrow, elongated shapes in satellite imagery, a plausible visual
explanation for the confusion rather than a modeling flaw.

Transfer learning made it possible to reach strong
performance on a domain quite different from the pre-trained model's
original training data (natural photographs vs. satellite imagery), using
a fraction of the data, compute, and training time that training a
comparable CNN from scratch would have required.